# Preparing Stocks

This notebook:

- loads the daily OHLCV data for all stock symbols,
- runs basic sanity checks on dates, missing values, and zeros,
- applies cleaning steps to the price columns (`Open`, `Close`, `Adj Close`, `Volume`).




# Import libraries And Useful Files

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from src.utils.loading_data import *
from src.utils.data_inspection import *
from src.utils.data_inspection_viz import *
from src.utils.metadata_enrich import *
from src.utils.data_cleaning import *


# Loading Stocks

In [3]:

# Folder containing all stock CSVs
stocks_dir = Path("data/stocks")

# Dictionary to hold all loaded DataFrames
all_stocks = {}

# Loop over every CSV file in the folder
for csv_file in sorted(stocks_dir.glob("*.csv")):
    symbol = csv_file.stem  # e.g. AAPL.csv → AAPL
    df = load_symbol_from_dir(stocks_dir, symbol)
    all_stocks[symbol] = df
    print(f"Loaded {symbol}: {len(df)} rows")

print(f"\n✅ Loaded {len(all_stocks)} stock symbols total.")


Loaded A: 5124 rows
Loaded AA: 14663 rows
Loaded AACG: 261 rows
Loaded AAL: 3653 rows
Loaded AAMC: 1837 rows
Loaded AAME: 10098 rows
Loaded AAN: 9431 rows
Loaded AAOI: 1640 rows
Loaded AAON: 6873 rows
Loaded AAP: 4616 rows
Loaded AAPL: 9909 rows
Loaded AAT: 2319 rows
Loaded AAU: 3594 rows
Loaded AAWW: 3622 rows
Loaded AAXN: 4729 rows
Loaded AB: 8056 rows
Loaded ABB: 4776 rows
Loaded ABBV: 1825 rows
Loaded ABC: 6293 rows
Loaded ABCB: 6514 rows
Loaded ABEO: 9968 rows
Loaded ABEV: 5808 rows
Loaded ABG: 4540 rows
Loaded ABIO: 5699 rows
Loaded ABM: 10098 rows
Loaded ABMD: 8236 rows
Loaded ABR: 4024 rows
Loaded ABT: 10098 rows
Loaded ABTX: 1128 rows
Loaded ABUS: 3194 rows
Loaded AC: 3075 rows
Loaded ACA: 353 rows
Loaded ACAD: 3989 rows
Loaded ACAM: 227 rows
Loaded ACAMU: 280 rows
Loaded ACB: 5055 rows
Loaded ACBI: 1111 rows
Loaded ACC: 3935 rows
Loaded ACCO: 3681 rows
Loaded ACEL: 269 rows
Loaded ACER: 4020 rows
Loaded ACGL: 6180 rows
Loaded ACGLO: 658 rows
Loaded ACGLP: 886 rows
Loaded ACH:

## 3. Date sanity checks

In this section we verify the time axis of our stock data:

- Check that dates inside each symbol are strictly increasing.
- Show the global date range for all stocks.
- Find symbols that end too early (before a chosen cutoff date).
- Plot the distribution of start dates across all symbols.
- Add the column weekday for further use

These checks help us detect symbols with strange or truncated histories
before we start cleaning.


In [4]:
print("\n🔍 Checking if dates of stocks are in increasing order...")

for symbol, df in all_stocks.items():
    if dates_in_increasing_order(df):
        print(f"✅ {symbol}: dates are in increasing order.")
    else:
        print(f"⚠️ {symbol}: dates are NOT in increasing order!")



🔍 Checking if dates of stocks are in increasing order...
✅ A: dates are in increasing order.
✅ AA: dates are in increasing order.
✅ AACG: dates are in increasing order.
✅ AAL: dates are in increasing order.
✅ AAMC: dates are in increasing order.
✅ AAME: dates are in increasing order.
✅ AAN: dates are in increasing order.
✅ AAOI: dates are in increasing order.
✅ AAON: dates are in increasing order.
✅ AAP: dates are in increasing order.
✅ AAPL: dates are in increasing order.
✅ AAT: dates are in increasing order.
✅ AAU: dates are in increasing order.
✅ AAWW: dates are in increasing order.
✅ AAXN: dates are in increasing order.
✅ AB: dates are in increasing order.
✅ ABB: dates are in increasing order.
✅ ABBV: dates are in increasing order.
✅ ABC: dates are in increasing order.
✅ ABCB: dates are in increasing order.
✅ ABEO: dates are in increasing order.
✅ ABEV: dates are in increasing order.
✅ ABG: dates are in increasing order.
✅ ABIO: dates are in increasing order.
✅ ABM: dates are in i

In [5]:
show_date_ranges(all_stocks, "stocks")



 Date ranges for all stocks:

A: from 1999-11-18 → 2020-04-01
AA: from 1962-01-02 → 2020-04-01
AACG: from 2019-03-11 → 2020-04-01
AAL: from 2005-09-27 → 2020-04-01
AAMC: from 2012-12-13 → 2020-04-01
AAME: from 1980-03-17 → 2020-04-01
AAN: from 1982-11-04 → 2020-04-01
AAOI: from 2013-09-26 → 2020-04-01
AAON: from 1992-12-16 → 2020-04-01
AAP: from 2001-11-29 → 2020-04-01
AAPL: from 1980-12-12 → 2020-04-01
AAT: from 2011-01-13 → 2020-04-01
AAU: from 2005-12-20 → 2020-04-01
AAWW: from 2005-11-09 → 2020-04-01
AAXN: from 2001-06-08 → 2020-04-01
AB: from 1988-04-14 → 2020-04-01
ABB: from 2001-04-06 → 2020-04-01
ABBV: from 2013-01-02 → 2020-04-01
ABC: from 1995-04-04 → 2020-04-01
ABCB: from 1994-05-19 → 2020-04-01
ABEO: from 1980-09-19 → 2020-04-01
ABEV: from 1997-03-05 → 2020-04-01
ABG: from 2002-03-21 → 2020-04-01
ABIO: from 1997-08-08 → 2020-04-01
ABM: from 1980-03-17 → 2020-04-01
ABMD: from 1987-07-29 → 2020-04-01
ABR: from 2004-04-07 → 2020-04-01
ABT: from 1980-03-17 → 2020-04-01
ABTX: f

In [ ]:
ends_before(all_stocks, "stocks", cutoff_date="2020-04-01")

In [ ]:
plot_start_date_distribution(all_stocks, "stocks")

In [6]:
def add_weekday_column(all_stocks, date_col="Date"):
    """
    Add a 'Weekday' column (0=Mon ... 4=Fri) right after the Date column.
    Modifies all_stocks in-place.
    """
    for symbol, df in all_stocks.items():

        # Ensure Date column is datetime
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

        # Compute weekday (0=Mon ... 6=Sun, but your data only has 0-4)
        weekday = df[date_col].dt.weekday

        # Position where 'Date' is located
        idx = df.columns.get_loc(date_col)

        # Insert new column right after Date
        df.insert(idx + 1, "Weekday", weekday)
        
add_weekday_column(all_stocks)



## 4. Missing data analysis

Here we look at missing values in the stock time series.

We:

- compute, for each symbol, the fraction of missing values;
- compute overall missing rates for the whole dataset;
- plot how the number of missing entries evolves over time.

This gives an overview of the data quality and shows if certain periods
have more missing data than others.


In [7]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)

 Missing rates per stock:
        Date  Weekday  Open  High  Low  Close  Adj Close  Volume  Symbol
Symbol                                                                  
A        0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AA       0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AACG     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAL      0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAMC     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0

 Overall missing rates for stocks:
           Total Missing Rate (%)
Date                     0.000000
Weekday                  0.000000
Open                     0.002682
High                     0.002682
Low                      0.002682
Close                    0.002682
Adj Close                0.002682
Volume                   0.002682
Symbol                   0.000000


In [ ]:
plot_missing_counts_over_time(all_stocks, label="Stocks")

In [8]:
def get_symbols_with_missing(all_stocks: dict, column: str):
    """
    Given a dict of DataFrames (key = symbol), return a list of symbols
    that have at least one missing value in the given column.
    """
    symbols = []

    for symbol, df in all_stocks.items():
        if column not in df.columns:
            raise ValueError(f"Column '{column}' not found in DataFrame for {symbol}")

        if df[column].isna().any():
            symbols.append(symbol)

    return sorted(symbols)


In [9]:
missing_open_symbols = get_symbols_with_missing(all_stocks, "Open")
missing_volume_symbols = get_symbols_with_missing(all_stocks, "Volume")
missing_high_symbols = get_symbols_with_missing(all_stocks, "High")
missing_low_symbols = get_symbols_with_missing(all_stocks, "Low")
missing_close_symbols = get_symbols_with_missing(all_stocks, "Close")
missing_adj_close_symbols = get_symbols_with_missing(all_stocks, "Adj Close")



In [10]:
print("Missing symbols for each column:\n")

print(f"Open:       {len(missing_open_symbols)} symbols")
print(missing_open_symbols[:20], "...\n" if len(missing_open_symbols) > 20 else "\n")

print(f"Volume:     {len(missing_volume_symbols)} symbols")
print(missing_volume_symbols[:20], "...\n" if len(missing_volume_symbols) > 20 else "\n")

print(f"High:       {len(missing_high_symbols)} symbols")
print(missing_high_symbols[:20], "...\n" if len(missing_high_symbols) > 20 else "\n")

print(f"Low:        {len(missing_low_symbols)} symbols")
print(missing_low_symbols[:20], "...\n" if len(missing_low_symbols) > 20 else "\n")

print(f"Close:      {len(missing_close_symbols)} symbols")
print(missing_close_symbols[:20], "...\n" if len(missing_close_symbols) > 20 else "\n")

print(f"Adj Close:  {len(missing_adj_close_symbols)} symbols")
print(missing_adj_close_symbols[:20], "...\n" if len(missing_adj_close_symbols) > 20 else "\n")


Missing symbols for each column:

Open:       256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', 'AMAT', 'AME', 'AMWD', 'AON', 'AOS', 'ASRVP', 'AT', 'AVT', 'AWF', 'AXP', 'BAC'] ...

Volume:     256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', 'AMAT', 'AME', 'AMWD', 'AON', 'AOS', 'ASRVP', 'AT', 'AVT', 'AWF', 'AXP', 'BAC'] ...

High:       256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', 'AMAT', 'AME', 'AMWD', 'AON', 'AOS', 'ASRVP', 'AT', 'AVT', 'AWF', 'AXP', 'BAC'] ...

Low:        256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', 'AMAT', 'AME', 'AMWD', 'AON', 'AOS', 'ASRVP', 'AT', 'AVT', 'AWF', 'AXP', 'BAC'] ...

Close:      256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', 'AMAT', 'AME', 'AMWD', 'AON', 'AOS', 'ASRVP', 'AT', 'AVT', 'AWF', 'AXP', 'BAC'] ...

Adj Close:  256 symbols
['ADP', 'AEM', 'AFB', 'AGE', 'AGYS', 'AIG', 'AKR', 'ALB', 'ALOT', '

In [11]:
# Put all lists into a dictionary for convenience
missing_dict = {
    "Open": missing_open_symbols,
    "Volume": missing_volume_symbols,
    "High": missing_high_symbols,
    "Low": missing_low_symbols,
    "Close": missing_close_symbols,
    "Adj Close": missing_adj_close_symbols
}

# Convert lists to sets for easy comparison
missing_sets = {col: set(syms) for col, syms in missing_dict.items()}

# Take the first set as reference
first_col = list(missing_sets.keys())[0]
reference_set = missing_sets[first_col]

# Check if all sets match the reference
all_same = all(missing_sets[col] == reference_set for col in missing_sets)

print("---------------------------------------------------")
if all_same:
    print("✅ All columns have missing values in EXACTLY the same symbols.")
else:
    print("❌ Columns do NOT have missing values in the same symbols.\n")

    # Show which ones differ
    print("Differences:")
    for col, s in missing_sets.items():
        if s != reference_set:
            print(f"- {col} differs from {first_col}")


---------------------------------------------------
✅ All columns have missing values in EXACTLY the same symbols.


# 5. Cleaning Missing Values

Since only 4.35% of the stocks' symbols have missing values, we'll just remove them

In [12]:
def remove_missing_symbols(all_stocks: dict, symbols_to_remove: list):
    """
    Removes the given symbols from the all_stocks dict (in-place).

    Parameters
    ----------
    all_stocks : dict
        Dictionary where keys = stock symbols and values = DataFrames.
    symbols_to_remove : list
        List of symbols to delete from all_stocks.
    """
    for sym in symbols_to_remove:
        if sym in all_stocks:
            del all_stocks[sym]

symbols_to_remove = missing_open_symbols  # since they are all the same
remove_missing_symbols(all_stocks, symbols_to_remove)
print("Remaining symbols:", len(all_stocks))
print("Removed symbols:", len(symbols_to_remove))
print(type(all_stocks))



Remaining symbols: 5628
Removed symbols: 256
<class 'dict'>


In [13]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)

 Missing rates per stock:
        Date  Weekday  Open  High  Low  Close  Adj Close  Volume  Symbol
Symbol                                                                  
A        0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AA       0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AACG     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAL      0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAMC     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0

 Overall missing rates for stocks:
           Total Missing Rate (%)
Date                          0.0
Weekday                       0.0
Open                          0.0
High                          0.0
Low                           0.0
Close                         0.0
Adj Close                     0.0
Volume                        0.0
Symbol                        0.0


## 6. Zero-value analysis

Zeros in price or volume columns can be problematic: they may represent bad data 

In this section we:

- measure how often each symbol has zero values;
- compute overall zero rates;
- plot the number of zeros over time for several columns:
  - `Open`
  - `Volume`
  - `Adj Close`
  - `Close`

This tells us whether zeros are rare anomalies or a systematic issue that needs cleaning.


In [14]:
zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")


print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)


 Zero value rates per symbol (STOCKS):

 Overall zero value rates (STOCKS):
           Total Zero Rate (%)
Weekday              18.850887
Open                  4.504569
High                  0.000081
Low                   0.000081
Close                 0.000076
Adj Close             0.000076
Volume                5.790265


In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Open")


In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Volume")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Adj Close")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Close")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="High")

In [ ]:
plot_zero_counts_over_time(all_stocks, label="Stocks",column="Low")

In [15]:
zero_symbols_high = get_symbols_with_zeros(all_stocks, "High")
print(zero_symbols_high)
zero_symbols_low = get_symbols_with_zeros(all_stocks, "Low")
print(zero_symbols_low)
print (zero_symbols_high == zero_symbols_low)

['APEX', 'ASYS', 'AXDX', 'HWKN', 'KERN', 'MAYS', 'NEOG', 'NRIM', 'PMD', 'PNRG', 'RMCF', 'SEED', 'STRL']
['APEX', 'ASYS', 'AXDX', 'HWKN', 'KERN', 'MAYS', 'NEOG', 'NRIM', 'PMD', 'PNRG', 'RMCF', 'SEED', 'STRL']
True


In [16]:
zero_symbols_vol = get_symbols_with_zeros(all_stocks, "Volume")
print(zero_symbols_vol)

['AA', 'AAMC', 'AAME', 'AAN', 'AAON', 'AAWW', 'ABCB', 'ABEO', 'ABEV', 'ABIO', 'ABM', 'ABMD', 'ABUS', 'AC', 'ACAD', 'ACAM', 'ACAMU', 'ACB', 'ACEL', 'ACER', 'ACGL', 'ACH', 'ACHC', 'ACHV', 'ACNB', 'ACOR', 'ACRS', 'ACRX', 'ACST', 'ACTT', 'ACTTU', 'ACU', 'ACV', 'ACY', 'ADC', 'ADES', 'ADMA', 'ADMP', 'ADNT', 'ADUS', 'ADXN', 'ADXS', 'AE', 'AEF', 'AEG', 'AEGN', 'AEHR', 'AEMD', 'AESE', 'AEY', 'AEYE', 'AEZS', 'AFC', 'AFG', 'AFH', 'AFI', 'AFL', 'AG', 'AGBA', 'AGBAU', 'AGCO', 'AGFS', 'AGI', 'AGM-A', 'AGM', 'AGMH', 'AGNCN', 'AGRX', 'AGX', 'AHCO', 'AHPI', 'AIC', 'AIH', 'AIM', 'AIN', 'AINC', 'AIO', 'AIR', 'AIRI', 'AIRT', 'AIRTP', 'AIT', 'AIW', 'AIZP', 'AJG', 'AJXA', 'AKCA', 'AKER', 'AKRX', 'AKTS', 'AKTX', 'ALAC', 'ALACR', 'ALACU', 'ALBO', 'ALC', 'ALCO', 'ALDX', 'ALE', 'ALEX', 'ALG', 'ALGR', 'ALGRU', 'ALJJ', 'ALKS', 'ALLT', 'ALLY', 'ALO', 'ALRS', 'ALSK', 'ALT', 'ALTG', 'ALTR', 'ALUS', 'ALX', 'ALXN', 'ALYA', 'AMAG', 'AMBO', 'AMCI', 'AMCIU', 'AMCR', 'AMD', 'AMED', 'AMEH', 'AMGN', 'AMHC', 'AMHCU', 'AMK', 

In [17]:
zero_symbols_close_adj = get_symbols_with_zeros(all_stocks, "Adj Close")
print(zero_symbols_close_adj)
zero_symbols_close = get_symbols_with_zeros(all_stocks, "Close")
print(zero_symbols_close)
print (zero_symbols_close_adj == zero_symbols_close)




['APEX', 'ASYS', 'AXDX', 'HWKN', 'MAYS', 'NEOG', 'NRIM', 'PMD', 'PNRG', 'RMCF', 'SEED', 'STRL']
['APEX', 'ASYS', 'AXDX', 'HWKN', 'MAYS', 'NEOG', 'NRIM', 'PMD', 'PNRG', 'RMCF', 'SEED', 'STRL']
True


## 6. Cleaning Zero Value Columns (`Open`, `Close`, `Adj Close`, `Volume` )

Using the information from the previous diagnostics, we now clean the price data:

1. Identify symbols where `Adj Close`, `Close`, `High`, `Low`  contain zeros.
2. Remove the worst symbols if they are too corrupted.
3. Fill remaining `Adj Close` gaps (NaNs) using a suitable rule.
4. Fix `Open` and `Close` when they are zero or inconsistent by using
   the `Adj Close` value from the previous or current day.
5. 
6. 


In [18]:
new_symbols_to_remove = list(set(zero_symbols_high)| set(zero_symbols_close))

print(len(new_symbols_to_remove))
print(new_symbols_to_remove)




13
['MAYS', 'APEX', 'SEED', 'AXDX', 'NRIM', 'KERN', 'ASYS', 'HWKN', 'STRL', 'NEOG', 'RMCF', 'PMD', 'PNRG']


In [19]:
remove_missing_symbols(all_stocks, new_symbols_to_remove)




In [20]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)

 Missing rates per stock:
        Date  Weekday  Open  High  Low  Close  Adj Close  Volume  Symbol
Symbol                                                                  
A        0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AA       0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AACG     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAL      0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAMC     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0

 Overall missing rates for stocks:
           Total Missing Rate (%)
Date                          0.0
Weekday                       0.0
Open                          0.0
High                          0.0
Low                           0.0
Close                         0.0
Adj Close                     0.0
Volume                        0.0
Symbol                        0.0


In [21]:
def fill_zero_open_with_prev_adj_close(all_stocks: dict):
    """
    For each symbol (DataFrame) in all_stocks:

    1. If the first row has Open == 0:
       - Store that day's Adj Close in a variable.
       - Drop the whole first row.
       - If the new first row still has Open == 0, replace it with the stored Adj Close.

    2. For the remaining rows where Open == 0:
       - Replace Open with the previous day's Adj Close.

    Assumptions:
    - No NaNs in 'Open' or 'Adj Close'.
    - We only want to fix zeros in 'Open'.

    Modifies all_stocks in-place.
    """
    for symbol, df in all_stocks.items():
        if df.empty:
            continue

        # Ensure sorted by Date (just in case)
        if "Date" in df.columns:
            df.sort_values("Date", inplace=True)

        if "Open" not in df.columns or "Adj Close" not in df.columns:
            continue

        # --- Step 1: handle first row if Open == 0 ---
        if df["Open"].iloc[0] == 0:
            first_idx = df.index[0]
            first_adj_close = df.loc[first_idx, "Adj Close"]

            # Drop the original first row completely
            df.drop(first_idx, inplace=True)

            # If everything is gone, move to next symbol
            if df.empty:
                continue

            # Re-check the new first row
            new_first_idx = df.index[0]
            if df.loc[new_first_idx, "Open"] == 0:
                df.loc[new_first_idx, "Open"] = first_adj_close

        # --- Step 2: fix remaining zeros using previous Adj Close ---
        open_zero_mask = (df["Open"] == 0)

        if not open_zero_mask.any():
            continue

        prev_adj_close = df["Adj Close"].shift(1)
        df.loc[open_zero_mask, "Open"] = prev_adj_close[open_zero_mask]


In [22]:
fill_zero_open_with_prev_adj_close(all_stocks)


In [23]:
zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")


print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)




 Zero value rates per symbol (STOCKS):

 Overall zero value rates (STOCKS):
           Total Zero Rate (%)
Weekday              18.850044
Open                  0.000000
High                  0.000000
Low                   0.000000
Close                 0.000000
Adj Close             0.000000
Volume                5.749932


In [24]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)

 Missing rates per stock:
        Date  Weekday  Open  High  Low  Close  Adj Close  Volume  Symbol
Symbol                                                                  
A        0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AA       0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AACG     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAL      0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0
AAMC     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0

 Overall missing rates for stocks:
           Total Missing Rate (%)
Date                          0.0
Weekday                       0.0
Open                          0.0
High                          0.0
Low                           0.0
Close                         0.0
Adj Close                     0.0
Volume                        0.0
Symbol                        0.0


In [29]:
def add_volatility(all_stocks):
    for symbol, df in all_stocks.items():
        df["Volatility"] = (df["High"] - df["Low"]) / df["Open"]


In [30]:
add_volatility(all_stocks)


In [31]:
def trim_leading_zero_volumes(all_stocks):
    """
    For each symbol, remove rows at the beginning where Volume == 0
    until the first non-zero Volume is found.
    """
    for symbol, df in all_stocks.items():
        if 'Volume' not in df.columns:
            raise ValueError(f"Symbol {symbol} has no 'Volume' column")

        nonzero_idx = df.index[df['Volume'] > 0]

        # if the stock has at least one non-zero volume
        if len(nonzero_idx) > 0:
            first_idx = nonzero_idx[0]
            all_stocks[symbol] = df.loc[first_idx:].copy()
        # else: all volumes are 0 → we leave it as is (or you can drop the symbol)
        

trim_leading_zero_volumes(all_stocks)

In [32]:
zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")


print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)




 Zero value rates per symbol (STOCKS):

 Overall zero value rates (STOCKS):
            Total Zero Rate (%)
Weekday               18.850129
Open                   0.000000
High                   0.000000
Low                    0.000000
Close                  0.000000
Adj Close              0.000000
Volume                 5.485643
Volatility             8.287598


In [33]:
import numpy as np
import pandas as pd

def fill_zero_volumes_past_only(all_stocks, n_same_weekday=10, vol_weight=0.7):
    """
    For each symbol DataFrame in all_stocks:

    - For each row with Volume == 0:
        baseline = median of past non-zero Volume on same Weekday
        if none: baseline = median of all past non-zero Volume
        filled_volume = baseline * (1 + vol_z * vol_weight)

    - vol_z is the z-score of Volatility for that symbol.
    - Uses ONLY past data (no future).

    Assumes columns: 'Weekday', 'Volume', 'Volatility'.
    Changes are IN PLACE.
    """

    for symbol, df in all_stocks.items():

        # --- check required columns ---
        for col in ['Weekday', 'Volume', 'Volatility']:
            if col not in df.columns:
                raise ValueError(f"{symbol}: missing column '{col}'")

        # --- volatility z-score (as a Series, no extra column) ---
        vol_mean = df['Volatility'].mean()
        vol_std  = df['Volatility'].std()

        if vol_std == 0 or np.isnan(vol_std):
            vol_z = pd.Series(0.0, index=df.index)
        else:
            vol_z = (df['Volatility'] - vol_mean) / vol_std

        # indices where Volume == 0
        zero_idx = df.index[df['Volume'] == 0]

        for idx in zero_idx:

            weekday = df.at[idx, 'Weekday']

            # all past rows (strictly before idx)
            past = df.loc[:idx-1]

            # 1) past non-zero same weekday
            past_same = past[(past['Weekday'] == weekday) &
                             (past['Volume'] > 0)]['Volume']

            if len(past_same) > 0:
                base_source = past_same.tail(n_same_weekday)
            else:
                # 2) fallback: all past non-zero volumes (any weekday)
                past_nonzero = past[past['Volume'] > 0]['Volume']
                if len(past_nonzero) == 0:
                    # no past info at all -> leave Volume at 0 for this row
                    continue
                base_source = past_nonzero.tail(n_same_weekday)

            baseline = base_source.median()

            # volatility adjustment
            adj = 1 + vol_z.at[idx] * vol_weight
            if adj < 0.01:
                adj = 0.01

            df.at[idx, 'Volume'] = baseline * adj


In [34]:
fill_zero_volumes_past_only(all_stocks)

/var/folders/z8/rtml_vtn47g7d8xsd6ctbs8m0000gn/T/ipykernel_27636/3275087269.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12677.310968894399' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'Volume'] = baseline * adj
/var/folders/z8/rtml_vtn47g7d8xsd6ctbs8m0000gn/T/ipykernel_27636/3275087269.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1498.802350083964' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'Volume'] = baseline * adj
/var/folders/z8/rtml_vtn47g7d8xsd6ctbs8m0000gn/T/ipykernel_27636/3275087269.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3321.2501398107875' has dtype incompatible with int64, please explicitl

In [35]:
zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")


print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)




 Zero value rates per symbol (STOCKS):

 Overall zero value rates (STOCKS):
            Total Zero Rate (%)
Weekday               18.850129
Open                   0.000000
High                   0.000000
Low                    0.000000
Close                  0.000000
Adj Close              0.000000
Volume                 0.000027
Volatility             8.287598


In [36]:
zero_symbols_vol = get_symbols_with_zeros(all_stocks, "Volume")
print(zero_symbols_vol)

['GHIV']


In [37]:
remove_missing_symbols(all_stocks, zero_symbols_vol)


# 7. Final Check

In [38]:
missing_per_stock, missing_stocks = missing_rates_from_dict(all_stocks)

print(" Missing rates per stock:")
print(missing_per_stock.head())

print("\n Overall missing rates for stocks:")
print(missing_stocks)

 Missing rates per stock:
        Date  Weekday  Open  High  Low  Close  Adj Close  Volume  Symbol  \
Symbol                                                                     
A        0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0   
AA       0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0   
AACG     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0   
AAL      0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0   
AAMC     0.0      0.0   0.0   0.0  0.0    0.0        0.0     0.0     0.0   

        Volatility  
Symbol              
A              0.0  
AA             0.0  
AACG           0.0  
AAL            0.0  
AAMC           0.0  

 Overall missing rates for stocks:
            Total Missing Rate (%)
Date                           0.0
Weekday                        0.0
Open                           0.0
High                           0.0
Low                            0.0
Close                          0.0
Adj Close    

In [39]:
zero_per_symbol_stocks, zero_overall_stocks = zero_rates_from_dict(all_stocks)
print(" Zero value rates per symbol (STOCKS):")


print("\n Overall zero value rates (STOCKS):")
print(zero_overall_stocks)




 Zero value rates per symbol (STOCKS):

 Overall zero value rates (STOCKS):
            Total Zero Rate (%)
Weekday               18.850130
Open                   0.000000
High                   0.000000
Low                    0.000000
Close                  0.000000
Adj Close              0.000000
Volume                 0.000000
Volatility             8.287573


In [40]:
import pickle

with open("stocks_dict.pkl", "wb") as f:
    pickle.dump(all_stocks, f)

In [41]:
with open("stocks_dict.pkl", "rb") as f:
    stocks_dict = pickle.load(f)

In [ ]:
from pathlib import Path
import pandas as pd

CLEANED_DIR = Path("data_cleaned")
PROCESSED_DIR = Path("data_processed")
CLEANED_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

def export_dict_to_cleaned_folder(data_dict, subfolder_name, date_col_name="Date"):
    """
    data_dict: dict[ticker -> DataFrame]
    subfolder_name: 'stocks' or 'etfs' etc.
    date_col_name: name of the date column, if it's currently the index we’ll reset it.
    """
    out_dir = CLEANED_DIR / subfolder_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for ticker, df in data_dict.items():
        df_t = df.copy()

        # If Date is index, bring it back as a column for saving
        if df_t.index.name == date_col_name or date_col_name not in df_t.columns:
            df_t = df_t.reset_index()

        # Make sure it's sorted
        df_t = df_t.sort_values(date_col_name)

        # One file per ticker, same spirit as raw data
        df_t.to_csv(out_dir / f"{ticker}.csv", index=False)

    print(f"Saved cleaned {subfolder_name} for {len(data_dict)} tickers in {out_dir}")


def dict_to_aggregated_parquet(data_dict, parquet_name, date_col_name="Date"):
    """
    Concatenate dict[ticker -> DataFrame] into one big DataFrame with columns:
    [Date, ticker, ... original columns ...]
    """
    all_dfs = []

    for ticker, df in data_dict.items():
        df_t = df.copy()

        # If Date is index, bring it back as column
        if df_t.index.name == date_col_name or date_col_name not in df_t.columns:
            df_t = df_t.reset_index()

        df_t[date_col_name] = pd.to_datetime(df_t[date_col_name])
        df_t = df_t.sort_values(date_col_name)
        df_t["ticker"] = ticker

        all_dfs.append(df_t)

    if not all_dfs:
        print(f"No data in dict for {parquet_name}, skipping parquet export.")
        return None

    df_all = pd.concat(all_dfs, ignore_index=True)
    out_path = PROCESSED_DIR / parquet_name
    df_all.to_parquet(out_path, index=False)

    print(f"Saved aggregated parquet: {out_path}")
    return df_all

In [ ]:
from pathlib import Path
import pandas as pd

CLEANED_DIR = Path("data_cleaned")
PROCESSED_DIR = Path("data_processed")
CLEANED_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(exist_ok=True)

# paste the two helper functions here (export_dict_to_cleaned_folder, dict_to_aggregated_parquet)

# 1) Save per-ticker cleaned files
export_dict_to_cleaned_folder(all_stocks, subfolder_name="stocks", date_col_name="Date")

# 2) Save aggregated parquet
stocks_clean = dict_to_aggregated_parquet(
    all_stocks,
    parquet_name="stocks_clean.parquet",
    date_col_name="Date",
)

# Quick summary
if stocks_clean is not None:
    print("\n=== Stocks summary ===")
    print(f"Tickers: {stocks_clean['ticker'].nunique()}")
    print(f"Date range: {stocks_clean['Date'].min()} → {stocks_clean['Date'].max()}")